# Databricks RAG with Vector Search — Technical Documentation Q&A

**Author:** Ravi Amaraweera — Senior Data Architect / Analytics Engineer  
**Stack:** Databricks Vector Search · Foundation Model APIs · LangChain (no Mosaic AI Agent Framework)  
**Data:** Apache Airflow open-source docs (Apache 2.0 licence)  
**Estimated cost:** \$5–\$15 within the Databricks free \$400 trial credits  

---

## What this notebook builds

A complete end-to-end **Retrieval-Augmented Generation (RAG)** pipeline:

```
Apache Airflow Docs
      |
      v  (scrape + chunk)
Delta Table  -->  Vector Search Index  -->  LangChain Retriever
                                                    |
                                        Question + Chunks --> LLM --> Answer
```

## Execution order

Run every cell **top-to-bottom**. Long-running steps (endpoint & index provisioning) have polling loops — just wait for them to complete.  
**Always run Cell 19 (Teardown) at the end of every session to stop billing.**

---
## Cell 1 — Create Unity Catalog and Schema

Unity Catalog organises all Databricks data assets in a three-level namespace: `catalog.schema.table`.  
- **Catalog** (`rag_portfolio`): the top-level container — equivalent to a database server  
- **Schema** (`doc_search`): a namespace inside the catalog — equivalent to a database  

`IF NOT EXISTS` makes this cell idempotent — safe to re-run without errors or data loss.

In [0]:
%sql
-- Create the top-level catalog that scopes all objects in this project.
-- Unity Catalog enforces access control at the catalog level.
CREATE CATALOG IF NOT EXISTS rag_portfolio;

-- Create the schema (database) where the Delta tables and vector index will live.
-- Using a dedicated schema keeps this project's assets isolated from other work.
CREATE SCHEMA IF NOT EXISTS rag_portfolio.doc_search;

---
## Cell 2 — Install Python Dependencies

Installs all required libraries into the cluster's virtual environment.  
The Databricks cluster already has PySpark, pandas, and numpy — only the LLM/RAG libraries need to be added.

**Expected behaviour:** the kernel will restart automatically after installation. This is normal.  
**After restart:** skip this cell and continue from Cell 3 onwards.

In [0]:
# Install LangChain and Databricks integrations.
# - langchain: the core orchestration framework for chaining LLM calls
# - langchain-text-splitters: utilities for chunking long documents
# - databricks-langchain: Databricks-specific retriever and LLM wrappers
# - requests + beautifulsoup4: HTTP client and HTML parser for doc scraping
%pip install -q langchain langchain-text-splitters requests beautifulsoup4 databricks-langchain

# Force a kernel restart so the newly installed packages are importable.
# Databricks will restart automatically -- this dbutils call ensures it happens inline.
dbutils.library.restartPython()

---
## Cell 3 — Scrape Apache Airflow Documentation

Downloads selected pages from the official Apache Airflow docs website.  
Each page is parsed with BeautifulSoup to extract only the main article body — 
navigation menus, headers, and footers are discarded to improve embedding quality.

**Data licence:** Apache Airflow documentation is published under the Apache 2.0 licence.  
**Customisation:** Replace this cell with any data loader (PDF, SharePoint, S3, etc.) — 
the rest of the pipeline is data-source agnostic.

In [0]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -- Data source configuration -----------------------------------------------
# Base URL for the stable release of the Apache Airflow documentation.
base_url = "https://airflow.apache.org/docs/apache-airflow/stable/"

# List of relative page paths to scrape.
# Add or remove pages here to expand/narrow the knowledge base.
pages = [
    "tutorial/index.html",
    "howto/operator/index.html",
    "concepts/dags.html",
    "concepts/tasks.html",
    "concepts/operators.html",
    "concepts/connections.html",
    "best-practices.html",
]

# -- Scraping loop -----------------------------------------------------------
documents = []  # will hold one dict per successfully scraped page

for page in pages:
    try:
        # Fetch the raw HTML for this page.
        resp = requests.get(base_url + page, timeout=30)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Locate the main content area.
        # Airflow docs use a <div class="body"> for article content;
        # fall back to <main> for any restructured pages.
        content_div = soup.find("div", class_="body") or soup.find("main")

        if content_div:
            text = content_div.get_text(separator="\n", strip=True)
            # Store the document with metadata that will be preserved through
            # chunking, indexing, and retrieval for source attribution.
            documents.append({
                "source": f"airflow-docs/{page}",
                "text": text,
                "doc_type": "airflow"
            })
            print(f"Downloaded: {page} ({len(text)} chars)")
        else:
            print(f"No content found: {page}")
    except Exception as e:
        # Non-fatal -- log the error and continue with remaining pages.
        print(f"Skipping {page}: {e}")

print(f"\nTotal pages downloaded: {len(documents)}")

---
## Cell 4 — Chunk the Documents

Splits each full-page document into smaller, overlapping text chunks suitable for embedding.  

**Why chunk?**
- Embedding models have token limits — a full HTML page often exceeds them
- Smaller, focused chunks produce more precise similarity search results
- Overlap ensures sentences split across a boundary remain semantically intact

`RecursiveCharacterTextSplitter` splits at natural boundaries in priority order:  
paragraph break → line break → sentence end → word boundary

In [0]:
# -- Chunking configuration --------------------------------------------------
# chunk_size: target character length per chunk (~1000 chars ~= 200-250 tokens)
# chunk_overlap: characters shared between consecutive chunks to preserve context
# separators: split priority -- prefer larger semantic boundaries first
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)

# -- Split all documents into chunks -----------------------------------------
chunks = []   # flat list of chunk dicts -- one entry per chunk
chunk_id = 0  # auto-incrementing integer primary key for the Delta Table

for doc in documents:
    splits = splitter.split_text(doc["text"])
    for split in splits:
        # Each chunk carries its source metadata forward for retrieval attribution.
        chunks.append({
            "id": chunk_id,             # primary key -- required for Delta Sync Index
            "text": split,              # the text that will be embedded
            "source": doc["source"],    # original page URL suffix for attribution
            "doc_type": doc["doc_type"],# document category for filtering
            "char_count": len(split)    # useful for monitoring chunk distribution
        })
        chunk_id += 1

avg_size = sum(c["char_count"] for c in chunks) / len(chunks)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunk size: {avg_size:.0f} chars")

---
## Cell 5 — Write Chunks to a Delta Table

Persists all chunks as a managed **Delta Table** in Unity Catalog.  
This table is the single source of truth for the vector index.  
Vector Search syncs from it automatically when new rows are added (via Change Data Feed).

**Delta format benefits:** ACID transactions, schema enforcement, time travel, Change Data Feed.

In [0]:
import pandas as pd

# -- Table configuration -----------------------------------------------------
# Fully-qualified three-part name: catalog.schema.table
TABLE_NAME = "rag_portfolio.doc_search.airflow_docs_chunks"

# Convert the Python list of dicts to a Spark DataFrame via pandas.
# Spark is used here for its native Delta write support.
df = spark.createDataFrame(pd.DataFrame(chunks))

(
    df.write
    .format("delta")             # write in Delta Lake format
    .mode("overwrite")           # replace any existing data on re-run
    .option("overwriteSchema", "true")  # allow schema changes on re-run
    .saveAsTable(TABLE_NAME)     # register the table in Unity Catalog
)

print(f"Wrote {df.count()} chunks to {TABLE_NAME}")

---
## Cell 6 — Inspect the Delta Table

Quick sanity check: displays the first few rows to confirm the write succeeded  
and the chunk text content looks correct before proceeding to indexing.

In [0]:
%sql
-- Display a sample of chunks to verify write success.
-- Check: id is sequential, text has meaningful content, source paths look correct.
SELECT id, LEFT(text, 120) AS text_preview, source, char_count
FROM rag_portfolio.doc_search.airflow_docs_chunks
ORDER BY id
LIMIT 10;

---
## Cell 7 — Enable Change Data Feed (CDF)

Change Data Feed is a Delta Lake table property that records every row-level change  
(insert, update, delete) in a separate `_change_data` log.

**Why this is required:**  
Databricks Vector Search uses CDF to perform **incremental synchronisation** —  
only changed rows are re-embedded and updated in the index, rather than  
re-processing all rows from scratch every time.

> **Important:** CDF must be enabled *before* the Delta Sync Index is created.

In [0]:
%sql
-- Enable Change Data Feed on the chunks table.
-- This property tells Delta Lake to maintain a change log alongside the table data.
-- The Vector Search service reads this log to keep the index in sync automatically.
ALTER TABLE rag_portfolio.doc_search.airflow_docs_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Verify CDF is now active.
DESCRIBE DETAIL rag_portfolio.doc_search.airflow_docs_chunks;

---
## Cell 8 — Create the Vector Search Endpoint

A **Vector Search endpoint** is the compute infrastructure that hosts your vector indexes  
and serves similarity search queries. It runs independently from the Spark cluster.

**Cost note:** The endpoint incurs **hourly charges** while running.  
Always run the Teardown cell (Cell 19) when finished.

**Wait time:** 5–10 minutes for provisioning.

In [0]:
from databricks.vector_search.client import VectorSearchClient

# -- Endpoint configuration --------------------------------------------------
ENDPOINT_NAME = "rag_portfolio_endpoint"

# Initialise the Vector Search client.
# Authentication is handled automatically via the cluster's Databricks token.
client = VectorSearchClient()

# STANDARD type is appropriate for development and low-to-medium query volumes.
# For production workloads with high QPS, use PERFORMANCE_OPTIMIZED.
try:
    client.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print(f"Endpoint '{ENDPOINT_NAME}' creation initiated. Provisioning takes 5-10 minutes.")
except Exception as e:
    # If the endpoint already exists from a previous run, this is non-fatal.
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists -- skipping creation.")
    else:
        raise  # re-raise unexpected errors

---
## Cell 9 — Wait for the Endpoint to Come Online

Endpoint provisioning is asynchronous. This cell polls every 30 seconds  
until the endpoint reports `ONLINE` status, then continues automatically.

In [0]:
import time

print(f"Waiting for endpoint '{ENDPOINT_NAME}' to come online...")

while True:
    endpoint_info = client.get_endpoint(ENDPOINT_NAME)
    status = endpoint_info.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"  Status: {status}")

    if status == "ONLINE":
        print(f"\nEndpoint '{ENDPOINT_NAME}' is ONLINE and ready.")
        break
    elif status in ("FAILED", "DELETED"):
        raise RuntimeError(f"Endpoint entered terminal state: {status}. Details: {endpoint_info}")

    # Poll every 30 seconds -- good balance between responsiveness and API rate usage.
    time.sleep(30)

---
## Cell 10 — Create the Delta Sync Vector Index

A **Delta Sync Index** links a Delta Table to the Vector Search endpoint.  
It reads the `text` column, passes each value through the embedding model,  
and stores the resulting vectors in the index.

**Key parameters:**
- `embedding_source_column`: the column whose text is embedded (our chunk text)
- `embedding_model_endpoint_name`: `databricks-gte-large-en` — Databricks-hosted, no API key needed
- `pipeline_type=TRIGGERED`: re-sync only when explicitly triggered

**Wait time:** 5–15 minutes for the initial embedding pass.

In [0]:
# -- Index configuration -----------------------------------------------------
INDEX_NAME = "rag_portfolio.doc_search.airflow_docs_index"

try:
    client.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        source_table_name=TABLE_NAME,      # the Delta Table created in Cell 5
        index_name=INDEX_NAME,             # fully-qualified index name in Unity Catalog
        primary_key="id",                  # must be unique per row; used for upserts
        embedding_source_column="text",    # the column containing text to embed
        # Foundation Model API embedding model -- hosted by Databricks, no API key needed.
        embedding_model_endpoint_name="databricks-gte-large-en",
        pipeline_type="TRIGGERED"          # sync on demand rather than continuously
    )
    print(f"Index '{INDEX_NAME}' creation initiated.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{INDEX_NAME}' already exists -- skipping creation.")
    else:
        raise

---
## Cell 11 — Wait for the Index to Be Ready

Polls every 30 seconds and checks both the status and `num_rows_indexed`  
to confirm all chunks were embedded before proceeding.

In [0]:
print(f"Waiting for index '{INDEX_NAME}' to be ready...")

while True:
    index_info = client.get_index(INDEX_NAME)
    status = index_info.get("status", {}).get("ready", False)
    detailed_state = index_info.get("status", {}).get("detailed_state", "UNKNOWN")
    rows_indexed = index_info.get("status", {}).get("indexed_row_count", 0)

    print(f"  State: {detailed_state} | Rows indexed: {rows_indexed}")

    # The index is ready when it is online AND has processed at least some rows.
    if status and rows_indexed > 0:
        print(f"\nIndex '{INDEX_NAME}' is ONLINE with {rows_indexed} rows indexed.")
        break
    elif detailed_state in ("FAILED",):
        raise RuntimeError(f"Index creation failed. Details: {index_info}")

    time.sleep(30)

---
## Cell 12 — Verify the Vector Index

Final pre-flight check: prints a summary of the index including row count,  
embedding model, and source table before wiring up the retriever.

In [0]:
index_info = client.get_index(INDEX_NAME)

print("=" * 60)
print("Vector Index Summary")
print("=" * 60)
print(f"  Name:          {index_info.get('name')}")
print(f"  Endpoint:      {index_info.get('endpoint_name')}")
print(f"  Source table:  {index_info.get('delta_sync_index_spec', {}).get('source_table')}")
print(f"  Status:        {index_info.get('status', {}).get('detailed_state')}")
print(f"  Rows indexed:  {index_info.get('status', {}).get('indexed_row_count')}")
print("=" * 60)

---
## Cell 13 — Build the LangChain Retriever

Wraps the Databricks Vector Search index in a LangChain `DatabricksVectorSearch` object  
and exposes it as a standard LangChain retriever.

**What is a retriever?**  
A LangChain retriever is an abstraction with a `.invoke(query)` method  
that converts a text query into a list of `Document` objects.  
`k=4`: fetch the top-4 most semantically similar chunks per query.

In [0]:
from databricks_langchain import DatabricksVectorSearch

# -- Connect LangChain to the Vector Search index ----------------------------
# DatabricksVectorSearch authenticates automatically via the cluster token
# and wraps the index in LangChain's VectorStore interface.
vectorstore = DatabricksVectorSearch(
    endpoint=ENDPOINT_NAME,
    index_name=INDEX_NAME,
    columns=["text", "source", "doc_type"]  # metadata columns to return with each result
)

# -- Convert to a LangChain Retriever ----------------------------------------
# search_type='similarity' uses cosine similarity over the embedding vectors.
# Increasing k gives the LLM more context but also increases prompt token cost.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# -- Quick smoke-test: run a direct retrieval query --------------------------
# This bypasses the LLM and directly shows which chunks would be retrieved.
test_query = "What is a DAG in Airflow?"
test_docs = retriever.invoke(test_query)

print(f"Test query: '{test_query}'")
print(f"Chunks retrieved: {len(test_docs)}")
for i, doc in enumerate(test_docs, 1):
    source = doc.metadata.get('source', 'unknown')
    print(f"  [{i}] {source}: {doc.page_content[:100]}...")

---
## Cell 14 — Build the RAG Chain

Assembles the full RAG pipeline using LangChain Expression Language (LCEL).  
LCEL uses the pipe operator (`|`) to chain components into a single runnable.

**Chain data flow:**  
User question → retriever → format_docs → prompt → LLM → StrOutputParser → answer string

**LLM:** `llama-4-maverick` via Databricks Foundation Model API — no external API key needed.

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from databricks_langchain import ChatDatabricks

# -- LLM configuration -------------------------------------------------------
# temperature=0: deterministic responses -- best for factual Q&A pipelines.
# max_tokens=512: caps response length; sufficient for Q&A answers.
llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",
    temperature=0,
    max_tokens=512
)

# -- System prompt -----------------------------------------------------------
# Constraining the LLM to only use the provided context chunks is a key
# hallucination-mitigation technique in RAG systems.
SYSTEM_PROMPT = """You are a technical assistant for Apache Airflow. \Answer the question using ONLY the context provided below. \If the answer is not contained in the context, say \"I don't know based on the provided documentation.\"

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

def format_docs(docs):
    """
    Concatenates retrieved Document objects into a single context string.
    Each chunk is separated by a double newline. The source path is appended
    to each chunk for attribution in the LLM context window.
    """
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        formatted.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# -- Assemble the RAG chain using LCEL ---------------------------------------
# RunnablePassthrough() passes the question through unchanged to the prompt.
chain = (
    {
        "context": retriever | format_docs,  # retrieve chunks, format as string
        "question": RunnablePassthrough()    # pass question unchanged to prompt
    }
    | prompt           # format system + user message with context and question
    | llm              # call the Foundation Model API
    | StrOutputParser()  # extract text content from the LLM response object
)

print("RAG chain assembled successfully.")

---
## Cell 15 — Run a Single Query (End-to-End Test)

Invokes the complete RAG chain with a single question.  
This is the first full end-to-end run: retrieve → prompt → LLM → answer.

In [0]:
# Change this question to explore different parts of the Airflow documentation.
test_question = "How do I create a DAG in Airflow?"

print(f"Question: {test_question}")
print("-" * 60)

# chain.invoke() runs all steps: retrieve --> format --> prompt --> LLM --> parse
answer = chain.invoke(test_question)
print(f"Answer:\n{answer}")

---
## Cell 16 — Batch Evaluation

Runs the chain against a diverse set of test questions to assess pipeline performance.  
Results are collected for analysis and persisted to Delta in Cell 18.

In [0]:
import datetime

# -- Evaluation question set -------------------------------------------------
# These questions cover different Airflow concepts to test retrieval breadth.
eval_questions = [
    "What is a DAG in Airflow?",
    "How do I schedule an Airflow DAG to run daily?",
    "What is the difference between a Task and an Operator?",
    "How do I set up a connection in Airflow?",
    "What are Airflow best practices for production deployments?",
]

eval_results = []

for i, question in enumerate(eval_questions, 1):
    print(f"[{i}/{len(eval_questions)}] {question}")

    # Retrieve chunks directly to capture metadata alongside the answer.
    retrieved_docs = retriever.invoke(question)
    sources = list({doc.metadata.get('source', 'unknown') for doc in retrieved_docs})

    answer = chain.invoke(question)

    eval_results.append({
        "question":   question,
        "answer":     answer,
        "num_chunks": len(retrieved_docs),
        "sources":    ", ".join(sources),
        "timestamp":  datetime.datetime.utcnow().isoformat()
    })
    print(f"  --> {answer[:120]}...\n")

print(f"Evaluation complete: {len(eval_results)} questions processed.")

---
## Cell 17 — Inspect Retrieved Context

For a selected question, prints retrieved chunks and the final answer side-by-side.  
Primary debugging tool for RAG quality issues:
- **Good answer + good chunks** → pipeline working correctly
- **Poor answer + good chunks** → LLM prompt or reasoning issue
- **Poor answer + poor chunks** → retrieval issue (chunking, embedding, or index quality)

In [0]:
# Change the index to inspect different evaluation questions (0-4).
inspect_idx = 0
inspect_question = eval_questions[inspect_idx]

print(f"Question: {inspect_question}")
print("=" * 60)

retrieved = retriever.invoke(inspect_question)
print(f"Retrieved {len(retrieved)} chunks:\n")

for i, doc in enumerate(retrieved, 1):
    source = doc.metadata.get('source', 'unknown')
    print(f"  Chunk {i} | Source: {source}")
    print(f"  Content: {doc.page_content[:200]}...")
    print()

print("=" * 60)
print("Generated Answer:")
print(eval_results[inspect_idx]['answer'])

---
## Cell 18 — Save Evaluation Results to Delta

Persists the evaluation results as a structured Delta Table.  
Creates an auditable, queryable record of RAG system performance  
that can be queried in Databricks SQL or visualised in a Dashboard.

In [0]:
EVAL_TABLE = "rag_portfolio.doc_search.eval_results"

# Convert eval results to a Spark DataFrame and write to Delta.
# append mode preserves results from previous evaluation runs.
eval_df = spark.createDataFrame(pd.DataFrame(eval_results))

(
    eval_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(EVAL_TABLE)
)

print(f"Saved {eval_df.count()} evaluation records to {EVAL_TABLE}")
eval_df.printSchema()

---
## Cell 19 — ⚠ Teardown (ALWAYS RUN BEFORE CLOSING)

**This is the most important cell in the notebook.**

The Vector Search endpoint is billed **per hour** while active.  
Failing to delete it will consume your Databricks credits continuously.

**Run this cell before:**
- Closing your browser tab
- Ending your work session
- Taking a break longer than 30 minutes

**Teardown order:** the index must be deleted before the endpoint.

In [0]:
# -- Teardown: delete index then endpoint ------------------------------------
# Step 1: Delete the vector index.
# This removes all stored embeddings and index metadata from the endpoint.
try:
    client.delete_index(INDEX_NAME)
    print(f"[1/2] Index '{INDEX_NAME}' deleted.")
except Exception as e:
    print(f"[1/2] Could not delete index (may already be deleted): {e}")

# Brief pause to allow index deletion to propagate before removing the endpoint.
time.sleep(10)

# Step 2: Delete the Vector Search endpoint.
# This deallocates the compute and stops all billing for the endpoint.
try:
    client.delete_endpoint(ENDPOINT_NAME)
    print(f"[2/2] Endpoint '{ENDPOINT_NAME}' deleted.")
except Exception as e:
    print(f"[2/2] Could not delete endpoint (may already be deleted): {e}")

print("\nTeardown complete. Billing for Vector Search has stopped.")
print("Note: Delta Tables remain in Unity Catalog. To delete them:")
print("  DROP TABLE rag_portfolio.doc_search.airflow_docs_chunks;")